# TrialLens inference benchmarks on Kaggle

GPU measurements for the report:

1. Task 4, attention: reference vs tiled online softmax vs PyTorch SDPA (accuracy, speed, memory)
2. Task 3, KV cache: cached vs uncached greedy decoding over several input and output lengths
3. Task 5, speculative decoding: Qwen2.5-0.5B draft for the Qwen2.5-1.5B target, first with the stock draft, then with a draft fine-tuned on the same training split

Set `TARGET_ADAPTER` in the next cell to the fine-tuned target adapter (a Hugging Face repo id or a path under `/kaggle/input`). Left as `None`, the base target is used; costs per token are the same, only acceptance rates on the criterion task change.

Same setup as the other notebooks: **GPU T4 x2** (only one GPU is used, so timings are not disturbed), Internet on, `GITHUB_TOKEN` attached. Run with Save Version, Save & Run All.

In [ ]:
TARGET_ADAPTER = None

import os
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
if not os.path.exists("/kaggle/working/TrialLens"):
    !git clone -q https://{token}@github.com/AbdullahRasheed452/TrialLens.git /kaggle/working/TrialLens
%cd /kaggle/working/TrialLens
!git pull -q
!git log --oneline -3
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
adapter_arg = f"--adapter {TARGET_ADAPTER}" if TARGET_ADAPTER else ""

def check():
    assert _exit_code == 0, f"command failed with exit code {_exit_code}"

In [ ]:
!pip uninstall -y -q torchao
!pip install -q transformers==5.20.2 peft==0.21.2 accelerate==1.15.0 datasets==5.1.0
check()
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python scripts/make_split.py
check()
!python -m pytest -q
check()

In [ ]:
# Task 4: attention accuracy and speed
!python scripts/bench_attention.py
check()

In [ ]:
# Task 3: KV-cached vs uncached greedy decoding
!python scripts/bench_kv_cache.py {adapter_arg}
check()

In [ ]:
# Task 5a: speculative decoding with the stock 0.5B draft
!python scripts/bench_speculative.py {adapter_arg} --out results/bench/speculative_cuda_stock_draft.json
check()

In [ ]:
# Task 5b: fine-tune the 0.5B draft on the same training split, then benchmark again
!python scripts/train_lora.py --model Qwen/Qwen2.5-0.5B-Instruct --run-name draft05 --epochs 4
check()
!python scripts/bench_speculative.py {adapter_arg} --draft-adapter checkpoints/draft05/lora-best --out results/bench/speculative_cuda_tuned_draft.json
check()

In [ ]:
!mkdir -p results/train/draft05 && cp checkpoints/draft05/train_log.json results/train/draft05/
!zip -qr /kaggle/working/triallens_bench.zip results/bench results/train/draft05 checkpoints/draft05/lora-best
check()
!ls -lh /kaggle/working/triallens_bench.zip